Delta Lake is an open-source storage layer that brings reliability, performance, and advanced features to data lakes. Built on top of Apache Spark, it enables ACID transactions, scalable metadata handling, and unifies straming and batch data processing. Delta Lake solves common data lake challenges such as data corruption, incosistent schemas, and lack of versioning.

**Key Features**

- ACID transactions for data reliability
- Schema enforcement and evolution
- Time Travel (data versioning)
- Upserts and deletes (MERGE INTO)
- Data compaction and optimization
- Native support for streaming and batch workloads

In [0]:
# we'll demonstrate each of these features with code and explanations

from pyspark.sql import Row

In [0]:
#Create initial dataframe with sample data
data = [Row(id = 1, name = 'Alice', age = 39), Row(id = 2,name = 'Bob', age = 32), Row(id = 3, name = 'Charlie', age = 3)]
df = spark.createDataFrame(data)


In [0]:
#overwrite and create a managed Delta table with the DataFrame in the workspace.delta_demo schema

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.delta_demo")

df.write.format('delta').mode('overwrite').saveAsTable('workspace.delta_demo.people_delta_demo')

In [0]:
#Read the Delta tale to verify creation and display its contents

delta_df = spark.read.table('workspace.delta_demo.people_delta_demo')
display(delta_df)

id,name,age
1,Alice,39
2,Bob,32
3,Charlie,3


In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS my_files.my_schema.my_table")

df.write.format('delta').mode('overwrite').save('/Volumes/my_files/my_schema/my_table')

df2 = spark.read.load("/Volumes/my_files/my_schema/my_table")
display(df2)

id,name,age
1,Alice,39
2,Bob,32
3,Charlie,3


In [0]:
#Schema Evolution

from pyspark.sql.functions import lit
from pyspark.sql import Row
#create dataframe with new rows and add 'country' column

data = [Row(id=1, name="John", age=20, country="USA"),
        Row(id=2, name="Mary", age=30, country="Canada")]
df = spark.createDataFrame(data).withColumn("country",lit("USA"))
display(df)        

id,name,age,country
1,John,20,USA
2,Mary,30,USA


In [0]:
#append new rows to the Delta table

df.write.format('delta').mode('append')\
.option('mergeSchema','true')\
    .saveAsTable('workspace.delta_demo.people_delta_demo')
#read the Delta table to verify the new rows and columns

delta_df = spark.read.table('workspace.delta_demo.people_delta_demo')
display(delta_df)

id,name,age,country
1,John,20,USA
2,Mary,30,USA
1,John,20,USA
2,Mary,30,USA
1,Alice,39,null
2,Bob,32,null
3,Charlie,3,null


In [0]:
%sql
--Time Travel
Select * from workspace.delta_demo.people_delta_demo version as of 2;

id,name,age
1,Alice,39
2,Bob,32
3,Charlie,3


In [0]:
%sql
select * from workspace.delta_demo.people_delta_demo;

id,name,age,country
1,John,20,USA
2,Mary,30,USA
1,John,20,USA
2,Mary,30,USA
1,Alice,39,null
2,Bob,32,null
3,Charlie,3,null


In [0]:
%sql
--Restore Table
RESTORE TABLE workspace.delta_demo.people_delta_demo TO VERSION AS OF 2


table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
1306,1,2,0,2934,0


In [0]:
%sql
select * from workspace.delta_demo.people_delta_demo

id,name,age
1,Alice,39
2,Bob,32
3,Charlie,3


In [0]:
%sql
describe history workspace.delta_demo.people_delta_demo;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2026-10-14T05:35:48.000Z,75252551884598,ng5030180@gmail.com,RESTORE,"Map(version -> 2, timestamp -> null)",null,List(2003565258279652),b07d9adb-5079-4c86-ab6a-ab6f0e6798d7,1014-053335-ocaq7cda-v2n,4,Serializable,false,"Map(numRestoredFiles -> 0, removedFilesSize -> 2934, numRemovedFiles -> 2, restoredFilesSize -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numOfFilesAfterRestore -> 1, tableSizeAfterRestore -> 1306)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
4,2026-10-13T10:58:38.000Z,75252551884598,ng5030180@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [], canMergeSchema -> true)",null,List(2003565258279652),c5ad0c38-c154-474e-bffc-9dcfc69a366b,1013-105555-juvdo4ya-v2n,3,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 1467)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
3,2026-10-13T10:58:29.000Z,75252551884598,ng5030180@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [], canMergeSchema -> true)",null,List(2003565258279652),10d1e757-870b-4ee8-bd5b-222d4ce3659b,1013-105555-juvdo4ya-v2n,2,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 1467)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
2,2026-10-13T05:50:49.000Z,75252551884598,ng5030180@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""e7264c35-a556-4cc1-9afc-43d4b43aea7b""}, statsOnLoad -> true)",null,List(2003565258279652),a9267539-3001-4ba1-9dce-4b1e4f535d40,1013-053259-cv89l058-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1306, numDeletionVectorsRemoved -> 0, numOutputRows -> 3, numOutputBytes -> 1306)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
1,2026-10-13T05:38:12.000Z,75252551884598,ng5030180@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""e7264c35-a556-4cc1-9afc-43d4b43aea7b""}, statsOnLoad -> true)",null,List(2003565258279652),3c62e5a0-b12d-442c-a3d7-a147526b05ea,1013-053259-cv89l058-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1306, numDeletionVectorsRemoved -> 0, numOutputRows -> 3, numOutputBytes -> 1306)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
0,2026-10-13T05:37:46.000Z,75252551884598,ng5030180@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""e7264c35-a556-4cc1-9afc-43d4b43aea7b""}, statsOnLoad -> true)",null,List(2003565258279652),5611fa89-26fd-46e8-945e-02d25288cb46,1013-053259-cv89l058-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 3, numOutputBytes -> 1306)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13


In [0]:
#Merge into
from pyspark.sql.functions import lit

In [0]:
#prepare new data for merge
from pyspark.sql import Row
data = [Row(id = 1, name = 'Tom', age = 28),Row(id = 7,name = 'Henry',age = 27),Row(id = 8, name = 'Jerry',age = 37)]

df = spark.createDataFrame(data)
display(df)

id,name,age
1,Tom,28
7,Henry,27
8,Jerry,37


In [0]:
 #create a temporary view for the new data

 df.createOrReplaceGlobalTempView("Updates")

 #Perform MERGE INTO for insert and update based on id column

 merge_sql = """
 MERGE INTO workspace.delta_demo.people_delta_demo t
 USING global_temp.Updates s
 ON t.id = s.id
 WHEN MATCHED THEN
 UPDATE SET t.name = s.name, t.age = s.age*
 WHEN NOT MATCHED
 THEN INSERT (id,name,age) VALUES (s.id,s.name,s.age)
 """

 spark.sql(merge_sql)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5681311556675333>, line 3
      1 #create a temporary view for the new data
----> 3 df.createOrReplaceGlobalTempView("Updates")
      5 #Perform MERGE INTO for insert and update based on id column
      7 merge_sql = """
      8 MERGE INTO workspace.delta_demo.people_delta_demo t
      9 USING global_temp.Updates s
   (...)     14 THEN INSERT (id,name,age) VALUES (s.id,s.name,s.age)
     15 """

File <command-5681311556675332>, line 5
      2 from pyspark.sql import Row
      3 data = [Row(id = 1, name = 'Tom', age = 28),Row(id = 7,name = 'Henry',age = 27),Row(id = 8, name = 'Jerry',age = 37)]
----> 5 df = spark.createDataFrame(data)
      6 display(df)

AnalysisException: [NOT_SUPPORTED_WITH_SERVERLESS] GLOBAL TEMPORARY VIEW is not supported on serverless compute. SQLSTATE: 0A000;
CreateViewCommand `Updates`, UTF8_BINARY,

In [0]:
#Remove country column and keep only id,name,age

result_df = spark.read.table(
    'workspace.delta_demo.people_delta_demo'
).select(
    'id',
    'name',
    'age'
)
display(result_df)

id,name,age
1,Alice,39
2,Bob,32
3,Charlie,3


In [0]:
%sql
describe history workspace.delta_demo.people_delta_demo;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2026-10-14T05:35:48.000Z,75252551884598,ng5030180@gmail.com,RESTORE,"Map(version -> 2, timestamp -> null)",null,List(2003565258279652),b07d9adb-5079-4c86-ab6a-ab6f0e6798d7,1014-053335-ocaq7cda-v2n,4,Serializable,false,"Map(numRestoredFiles -> 0, removedFilesSize -> 2934, numRemovedFiles -> 2, restoredFilesSize -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numOfFilesAfterRestore -> 1, tableSizeAfterRestore -> 1306)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
4,2026-10-13T10:58:38.000Z,75252551884598,ng5030180@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [], canMergeSchema -> true)",null,List(2003565258279652),c5ad0c38-c154-474e-bffc-9dcfc69a366b,1013-105555-juvdo4ya-v2n,3,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 1467)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
3,2026-10-13T10:58:29.000Z,75252551884598,ng5030180@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [], canMergeSchema -> true)",null,List(2003565258279652),10d1e757-870b-4ee8-bd5b-222d4ce3659b,1013-105555-juvdo4ya-v2n,2,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 1467)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
2,2026-10-13T05:50:49.000Z,75252551884598,ng5030180@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""e7264c35-a556-4cc1-9afc-43d4b43aea7b""}, statsOnLoad -> true)",null,List(2003565258279652),a9267539-3001-4ba1-9dce-4b1e4f535d40,1013-053259-cv89l058-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1306, numDeletionVectorsRemoved -> 0, numOutputRows -> 3, numOutputBytes -> 1306)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
1,2026-10-13T05:38:12.000Z,75252551884598,ng5030180@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""e7264c35-a556-4cc1-9afc-43d4b43aea7b""}, statsOnLoad -> true)",null,List(2003565258279652),3c62e5a0-b12d-442c-a3d7-a147526b05ea,1013-053259-cv89l058-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1306, numDeletionVectorsRemoved -> 0, numOutputRows -> 3, numOutputBytes -> 1306)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
0,2026-10-13T05:37:46.000Z,75252551884598,ng5030180@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""e7264c35-a556-4cc1-9afc-43d4b43aea7b""}, statsOnLoad -> true)",null,List(2003565258279652),5611fa89-26fd-46e8-945e-02d25288cb46,1013-053259-cv89l058-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 3, numOutputBytes -> 1306)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13


***Delta Lake Maintenance Operations***

- Optimizing Tables: Use the OPTIMIZE command to compact small files and improve query performance
- ZORDER optimization: Apply ZORDER BY to colocate related data, enabling efficient data skipping and faster queries
- Running VACUUM: Remove obsolete files and reclaim storage space by executing the VACUUM command.

These features help maintain efficient, performant Delta tables in your data lake.

In [0]:
%sql
drop table if exists workspace.delta_demo.people_delta_demo

In [0]:
from pyspark.sql import Row

#create initial DataFrame with sample data

data  = [Row(id = 1,name = 'Alice', age = 30),Row(id = 2,name = 'Bob', age = 25),Row(id = 3,name = 'Charlie', age = 35)]
df = spark.createDataFrame(data)

#overwrite and create a managed Delta table with the DataFrame in the training.delta_demo schema

df.write.format('delta').mode('overwrite').saveAsTable('workspace.delta_demo.people_delta_demo')

#Read the delta table to verify creation and display its contents

delta_df = spark.read.table('workspace.delta_demo.people_delta_demo')
display(delta_df)

id,name,age
1,Alice,30
2,Bob,25
3,Charlie,35


In [0]:
df.write.format('delta').mode('append').save("/Volumes/my_files/my_schema/my_table")

In [0]:
df2 = spark.read.format('delta').load("/Volumes/my_files/my_schema/my_table")
display(df2)

id,name,age
1,Alice,30
2,Bob,25
3,Charlie,35
1,Alice,39
2,Bob,32
3,Charlie,3


In [0]:
%sql
--Optimize

optimize "/Volumes/my_files/my_schema/my_table"

path,metrics
dbfs:/Volumes/my_files/my_schema/my_table,"List(1, 2, List(1359, 1359, 1359.0, 1, 1359), List(1287, 1287, 1287.0, 2, 2574), 0, null, null, 0, 1, 2, 0, true, 0, 0, 1791962704705, 1791962708521, 8, 1, null, List(0, 0), null, 3, 3, 501, 0, null, null, 0)"


In [0]:
%sql
optimize workspace.delta_demo.people_delta_demo

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1791962795218, 1791962795943, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, null, null, 0)"


ZORDER

In [0]:
%sql
optimize "/Volumes/my_files/my_schema/my_table" zorder by id

path,metrics
dbfs:/Volumes/my_files/my_schema/my_table,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 1359), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1791962860727, 1791962861773, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, null, null, 0)"


In [0]:
%sql
optimize workspace.delta_demo.people_delta_demo zorder by id

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 1305), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1791962935851, 1791962936626, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, null, null, 0)"


In [0]:
%sql
DESCRIBE HISTORY workspace.delta_demo.people_delta_demo;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-10-14T07:09:51.000Z,75252551884598,ng5030180@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""66c1ad91-58d0-441c-8d27-a7c05afe4ac3""}, statsOnLoad -> true)",null,List(2003565258279652),e1355007-96fa-45cd-8b26-ef8d6e86d671,1014-065903-2no8mln8-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 3, numOutputBytes -> 1305)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13


In [0]:
%sql
--vacuum

vacuum "/Volumes/my_files/my_schema/my_table"

path
dbfs:/Volumes/my_files/my_schema/my_table


**Cloning**

Creates a new Delta table instantly from an existing table with/without duplication

- Shallow Clone -> Creates a quick copy that shares the same underlying data files but has its own metadata
- Deep Clone -> Creates a full, independent copy of both metadata and all data files

In [0]:
#create dataset

from pyspark.sql import Row

#create initial dataframe with sample dta

data = [Row(id = 1 , name = 'Alice', age = 30), Row(id = 2, name = 'Bob', age = 25), Row(id = 3, name = 'Charlie', age = 35)]

df = spark.createDataFrame(data)

In [0]:
#overwrite and create a managed Delta table with the DataFrame in the training.delta_demo shcema

df.write.format('delta').mode('overwrite').saveAsTable('workspace.delta_demo.source_table')

#Read the Delta table to verify creation and display its contents

delta_df = spark.read.table('workspace.delta_demo.source_table')
display(delta_df)

id,name,age
1,Alice,30
2,Bob,25
3,Charlie,35


In [0]:
df.write.format('delta').mode('append').saveAsTable('workspace.delta_demo.source_table')

In [0]:
df2 = spark.read.table('workspace.delta_demo.source_table')
display(df2)

id,name,age
1,Alice,30
2,Bob,25
3,Charlie,35
1,Alice,30
2,Bob,25
3,Charlie,35


Shallow Clone

In [0]:
%sql
Create or replace table workspace.delta_demo.source_table_clone SHALLOW CLONE workspace.delta_demo.source_table;

Deep Clone

In [0]:
%sql
CREATE or REPLACE TABLE workspace.delta_demo.target_deep DEEP CLONE workspace.delta_demo.source_table;

**Partitioning & Liquid Clustering**

- Partitioning -> Organizes data into directory-based groups to speed up queries by pruning entire folders

- Liquid Clustering -> Continously reorganizes data into optimized file-level clusters without fixed partitions for faster, more balanced performance

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.delta_demo.sales_partitioned(
    order_id STRING,
    order_date DATE,
    country STRING,
    category STRING,
    amount DOUBLE
)

USING DELTA 
PARTITIONED BY (country, category)

In [0]:
%sql
-- Insert data

INSERT INTO workspace.delta_demo.sales_partitioned VALUES

("1", "2026-01-01","USA","Electronics",1200.50),
("2", "2026-01-02","Eng","Clothing",450.20),
("3", "2026-01-03","SAF","Clothing",1000.00),
("4", "2026-01-04","India","Furniture",1100.90);

num_affected_rows,num_inserted_rows
4,4


In [0]:
df = spark.read.table("workspace.delta_demo.sales_partitioned")

df.write.format('delta').mode('overwrite').partitionBy("country","category").saveAsTable('workspace.delta_demo.sales_partitioned_copy')

Liquid Clustering

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.delta_demo.sales_clustered(
    order_id STRING,
    order_date DATE,
    country STRING,
    category STRING,
    amount DOUBLE
)

USING DELTA 
CLUSTER BY (country, category)

In [0]:
%sql
-- Insert data

INSERT INTO workspace.delta_demo.sales_clustered VALUES

("1", "2026-01-01","USA","Electronics",1200.50),
("2", "2026-01-02","Eng","Clothing",450.20),
("3", "2026-01-03","SAF","Clothing",1000.00),
("4", "2026-01-04","India","Furniture",1100.90);

num_affected_rows,num_inserted_rows
4,4


In [0]:
%sql
optimize workspace.delta_demo.sales_clustered

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 0, false, 0, 0, 1792215310952, 1792215315719, 8, 0, null, List(0, 0), null, 5, 5, 0, 0, List(1764, true, false, false, null, null, null, null, 0, 0, 0, 0, 1, 1764, 1764, null, log, 16777216, 67108864, 4, 0, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(177, 112, 0, 0, 0, 0, 1677), 2, 1, 5, sizeAware, defaultSplitStrategy, null, false, 0, null, false, 0, 0, 0, null, null, null, -1.0, n/a, 0, 0, false, false, -1.0, n/a), null, 0)"
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1792215315778, 1792215317624, 8, 0, null, List(0, 0), null, 5, 5, 0, 0, List(1764, false, false, false, null, null, null, post-optimize-compaction, 0, 0, 0, 0, 0, 0, 0, null, null, 33554432, 67108864, 0, 0, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(0, 0, 342, 0, 0, 0, 0), 15, 1, 1, null, null, null, false, 0, null, false, 0, 0, 0, null, null, null, -1.0, n/a, 0, 0, false, false, -1.0, n/a), null, 0)"


**Change Data Feed**

In [0]:
%sql
CREATE TABLE workspace.delta_demo.people(first_name STRING, age LONG) USING delta TBLPROPERTIES (delta.enableChangeDataFeed = true)

In [0]:
#Insert data

df = spark.createDataFrame([("Bob",23),
                            ("Sue",25),
                            ("Jim",27)]).toDF(
                                "first_name","age"
                            )
df.write.mode("append").format("delta").saveAsTable("workspace.delta_demo.people")

In [0]:
%sql
select * from workspace.delta_demo.people;

first_name,age
Bob,23
Sue,25
Jim,27


In [0]:
%sql
delete from workspace.delta_demo.people where first_name = 'Bob'

num_affected_rows
1


In [0]:
%sql
update workspace.delta_demo.people
set age = 30
where first_name = 'Jim'

num_affected_rows
1


In [0]:
#read CDF
df = spark.read.format("delta")\
    .option("readChangeFeed","true")\
        .option("startingVersion",0)\
            .table("workspace.delta_demo.people")

display(df)

first_name,age,_change_type,_commit_version,_commit_timestamp
Jim,27,update_preimage,4,2026-10-17T07:53:09.000Z
Jim,30,update_postimage,4,2026-10-17T07:53:09.000Z
Bob,23,insert,1,2026-10-17T07:51:48.000Z
Sue,25,insert,1,2026-10-17T07:51:48.000Z
Jim,27,insert,1,2026-10-17T07:51:48.000Z
Bob,23,delete,2,2026-10-17T07:52:40.000Z


In [0]:
%sql
--Alter Table Properties

ALTER TABLE workspace.delta_demo.people SET TBLPROPERTIES (delta.enableChangeDataFeed = TRUE)